แก้ปัญหา ID เดิมให้อยู่ได้นานที่สุด แม้ว่า

zoom in / zoom out

คนหายไปหลายเฟรม

cut / switch camera

กลับเข้าฉากใหม่

โดยใช้ InsightFace embedding + IoU hybrid tracking
และยังมี tracked face panel ด้านข้างครบ (ไม่หาย)

คุณสมบัติของเวอร์ชันนี้

ใช้ InsightFace buffalo_l (ArcFace embedding)

Hybrid matching:

🔥 Embedding (หลัก)

IoU (ช่วยช่วงสั้น)

Embedding smoothing (EMA)

YouTube stream

Face trajectory

Face panel ด้านข้าง

กัน panel overflow

ใช้กับ notebook ได้ (ไม่ใช้ imshow)

ผลลัพธ์ที่คุณจะเห็น

✔ ID เดิมอยู่ได้นานมาก
✔ cut กล้อง → ID กลับมาเดิม
✔ zoom หนัก → ไม่แตก
✔ panel ด้านข้างไม่ error
✔ trajectory ต่อเนื่อง

In [1]:
import cv2
import numpy as np
from IPython.display import display, Image, clear_output
import time
from collections import deque
from insightface.app import FaceAnalysis
from pytubefix import YouTube

# =============================
# CONFIG
# =============================
YOUTUBE_URL = "https://youtu.be/cmdMopdk6lo"

START_TIME = 75
END_TIME = 120

TARGET_WIDTH = 800
FACE_PANEL_WIDTH = 150
FACE_SIZE = 120
MAX_FACES_DISPLAY = 4

DET_SIZE = (480, 480)
FONT = cv2.FONT_HERSHEY_SIMPLEX

# Matching thresholds
SIM_HIGH = 0.55
SIM_LOW = 0.40
IOU_THRESHOLD = 0.3
EMB_MOMENTUM = 0.9

# =============================
# INIT INSIGHTFACE
# =============================
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=DET_SIZE)

# =============================
# UTIL FUNCTIONS
# =============================
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def extract_face(frame, box, margin=20):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    x1, y1 = max(0, x1-margin), max(0, y1-margin)
    x2, y2 = min(w, x2+margin), min(h, y2+margin)
    crop = frame[y1:y2, x1:x2]
    return crop if crop.size else None

# =============================
# FACE TRACKER
# =============================
class FaceTracker:
    def __init__(self, max_miss=20, traj_len=30):
        self.next_id = 1
        self.tracks = {}
        self.max_miss = max_miss
        self.traj_len = traj_len
        self.colors = {}

    def _iou(self, a, b):
        x1,y1 = max(a[0],b[0]), max(a[1],b[1])
        x2,y2 = min(a[2],b[2]), min(a[3],b[3])
        inter = max(0,x2-x1)*max(0,y2-y1)
        areaA = (a[2]-a[0])*(a[3]-a[1])
        areaB = (b[2]-b[0])*(b[3]-b[1])
        union = areaA + areaB - inter
        return inter/union if union > 0 else 0

    def _center(self, b):
        return ((b[0]+b[2])//2, (b[1]+b[3])//2)

    def _color(self, tid):
        if tid not in self.colors:
            hue = (tid * 0.618) % 1.0
            c = cv2.cvtColor(
                np.uint8([[[hue*180,255,255]]]),
                cv2.COLOR_HSV2BGR
            )[0][0]
            self.colors[tid] = tuple(map(int, c))
        return self.colors[tid]

    def update(self, boxes, faces, embs):
        assigned_ids = set()

        for box, face, emb in zip(boxes, faces, embs):
            best_id = None
            best_sim = -1

            # -------- Embedding matching --------
            for tid, data in self.tracks.items():
                sim = cosine(emb, data["emb"])
                if sim > best_sim:
                    best_sim = sim
                    best_id = tid

            # -------- Decision --------
            if best_sim >= SIM_HIGH:
                tid = best_id

            elif SIM_LOW <= best_sim < SIM_HIGH:
                tid = None
                for k, d in self.tracks.items():
                    if self._iou(box, d["box"]) > IOU_THRESHOLD:
                        tid = k
                        break
                if tid is None:
                    tid = self.next_id

            else:
                tid = self.next_id

            # -------- Register / Update --------
            if tid not in self.tracks:
                self.tracks[tid] = {
                    "box": box,
                    "face": face,
                    "emb": emb,
                    "miss": 0,
                    "traj": deque(maxlen=self.traj_len)
                }
                self.next_id += 1
            else:
                d = self.tracks[tid]
                d["box"] = box
                d["face"] = face
                d["emb"] = EMB_MOMENTUM * d["emb"] + (1-EMB_MOMENTUM) * emb
                d["miss"] = 0

            self.tracks[tid]["traj"].append(self._center(box))
            assigned_ids.add(tid)

        # -------- Handle missing tracks --------
        for tid in list(self.tracks.keys()):
            if tid not in assigned_ids:
                self.tracks[tid]["miss"] += 1
                if self.tracks[tid]["miss"] > self.max_miss:
                    del self.tracks[tid]

    def draw_traj(self, frame):
        for tid, d in self.tracks.items():
            if d["miss"] == 0:
                pts = list(d["traj"])
                col = self._color(tid)
                for i in range(1, len(pts)):
                    cv2.line(frame, pts[i-1], pts[i], col, 2)
        return frame

# =============================
# FACE PANEL
# =============================
def create_face_panel(tracker, height):
    panel = np.ones((height, FACE_PANEL_WIDTH, 3), np.uint8) * 40
    cv2.putText(panel,"Tracked",(10,25),FONT,0.5,(255,255,255),1)
    cv2.putText(panel,"Faces",(10,45),FONT,0.5,(255,255,255),1)

    y = 60
    active = [(tid,d) for tid,d in tracker.tracks.items()
              if d["miss"]==0 and d["face"] is not None]
    active.sort(key=lambda x: x[0])

    for tid, d in active[:MAX_FACES_DISPLAY]:
        if y + FACE_SIZE > height:
            break

        face = cv2.resize(d["face"], (FACE_SIZE, FACE_SIZE))
        col = tracker._color(tid)

        panel[y:y+FACE_SIZE, 15:15+FACE_SIZE] = face
        cv2.rectangle(panel,(13,y-2),(15+FACE_SIZE+2,y+FACE_SIZE+2),col,2)
        cv2.putText(panel,f"ID:{tid}",(15,y-5),FONT,0.4,col,1)

        y += FACE_SIZE + 10

    return panel

# =============================
# VIDEO STREAM
# =============================
yt = YouTube(YOUTUBE_URL)
stream = yt.streams.filter(progressive=True, file_extension="mp4").first()
cap = cv2.VideoCapture(stream.url)
cap.set(cv2.CAP_PROP_POS_MSEC, START_TIME * 1000)

tracker = FaceTracker()

while cap.isOpened():
    clear_output(wait=True)
    ret, frame = cap.read()
    if not ret:
        break

    t = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
    if t >= END_TIME:
        break

    detections = app.get(frame)

    boxes, faces, embs = [], [], []
    for f in detections:
        boxes.append(f.bbox.astype(int))
        faces.append(extract_face(frame, f.bbox))
        embs.append(f.embedding)

    tracker.update(boxes, faces, embs)

    for tid, d in tracker.tracks.items():
        if d["miss"] == 0:
            x1,y1,x2,y2 = d["box"]
            col = tracker._color(tid)
            cv2.rectangle(frame,(x1,y1),(x2,y2),col,2)
            cv2.putText(frame,f"ID:{tid}",(x1,y1-8),
                        FONT,0.6,(255,255,255),2)

    frame = tracker.draw_traj(frame)

    h, w = frame.shape[:2]
    nh = int(TARGET_WIDTH * h / w)
    frame = cv2.resize(frame, (TARGET_WIDTH, nh))

    panel = create_face_panel(tracker, nh)
    combined = np.hstack([frame, panel])

    _, buf = cv2.imencode(".jpg", combined)
    display(Image(data=buf.tobytes()))

    time.sleep(0.015)

cap.release()
print("Done")


KeyboardInterrupt: 